# Week 44, Day 4: The basal ganglia: actor, critic, Go and NoGo

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w044/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Q-learning in a gridworld

Complete qlearn: choose epsilon-greedily, step, compute the Q-learning TD error (no future value if the episode ended) and update Q. Record the number of steps in each episode.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2005)
S, A, GOAL = 25, 4, 24

def step(s, a):
    x, y = divmod(s, 5)
    x = max(0, x - 1) if a == 0 else min(4, x + 1) if a == 1 else x
    y = max(0, y - 1) if a == 2 else min(4, y + 1) if a == 3 else y
    s2 = 5 * x + y
    return s2, float(s2 == GOAL), s2 == GOAL

def qlearn(episodes=200, alpha=0.5, gamma=0.9, eps=0.1):
    Q = np.zeros((S, A)); lengths = []
    for k in range(episodes):
        s, n = 0, 0
        while True:
            a = r.integers(A) if r.random() < eps else np.argmax(Q[s] + 1e-9 * r.random(A))
            s2, rew, done = step(s, a)
            ...
            s, n = s2, n + 1
            if done or n > 200:
                break
        lengths.append(n)
    return Q, np.array(lengths)

Q, L_q = qlearn()
print(f"steps in the first five episodes {L_q[:5]}, mean of the last 20 {L_q[-20:].mean():.1f}; V(start) = {Q.max(1)[0]:.3f} (0.9^7 = {0.9 ** 7:.3f})")
plt.imshow(Q.max(1).reshape(5, 5)); plt.colorbar(); plt.title("max_a Q(s, a)"); plt.show()

In [ ]:
import numpy as np
def test_short():
    "after 200 episodes the average path in the last 20 is under 11 steps (the shortest is 8)"
    assert L_q[-20:].mean() < 11
def test_value():
    "the learned value of the start state is close to gamma to the 7th power"
    assert abs(Q.max(1)[0] - 0.9 ** 7) < 0.05

test_short()
test_value()
print("All checks passed")

### Exercise: An actor-critic on the same grid

Complete actor_critic: choose by softmax over H[s], step, compute delta = r + gamma V(s') - V(s) (no future value at the goal), and update both V[s] and H[s, a] with delta.

In [ ]:
def actor_critic(episodes=200, a_c=0.5, a_a=0.5, gamma=0.9):
    V = np.zeros(S); H = np.zeros((S, A)); lengths = []
    for k in range(episodes):
        s, n = 0, 0
        while True:
            p = np.exp(H[s] - H[s].max()); p /= p.sum()
            a = r.choice(A, p=p)
            s2, rew, done = step(s, a)
            ...
            s, n = s2, n + 1
            if done or n > 200:
                break
        lengths.append(n)
    return V, H, np.array(lengths)

V_ac, H_ac, L_ac = actor_critic()
print(f"actor-critic: first five episodes {L_ac[:5]}, mean of the last 20 {L_ac[-20:].mean():.1f}")
plt.plot(L_q, label="Q-learning"); plt.plot(L_ac, label="actor-critic"); plt.yscale("log"); plt.xlabel("episode"); plt.ylabel("steps"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_learns():
    "the actor-critic's last 20 episodes average under 13 steps"
    assert L_ac[-20:].mean() < 13
def test_critic():
    "the critic's value rises towards the goal along the bottom row"
    v = V_ac.reshape(5, 5)[4]
    assert v[4] == 0 and v[3] > v[1]

test_learns()
test_critic()
print("All checks passed")

### Exercise: Optimism and pessimism

For each learner, update V towards outcomes drawn with p = 0.8 for 20,000 trials, using alpha_plus when the error is positive and alpha_minus otherwise, and average V over the last 10,000 trials. Compare with the formula.

In [ ]:
g = np.random.default_rng(2004)
learners = {"optimist": (0.3, 0.05), "balanced": (0.1, 0.1), "pessimist": (0.05, 0.3)}
p = 0.8
sim, theory = {}, {}
for name, (ap, am) in learners.items():
    V, acc = 0.5, []
    for t in range(20000):
        out = float(g.random() < p)
        ...
        if t >= 10000:
            acc.append(V)
    sim[name] = np.mean(acc)
    theory[name] = p * ap / (p * ap + (1 - p) * am)
    print(f"{name:9s} simulated {sim[name]:.3f}, formula {theory[name]:.3f}")

In [ ]:
import numpy as np
def test_formula():
    "simulated values match the fixed-point formula within 0.03"
    assert all(abs(sim[k] - theory[k]) < 0.03 for k in sim)
def test_order():
    "the optimist values the option above 0.8 and the pessimist below"
    assert sim["optimist"] > 0.8 > sim["pessimist"]

test_formula()
test_order()
print("All checks passed")